# Final Swin Model Test And Baseline Comparison

This notebook evaluates saved checkpoints without any training. It compares three Swin-based levels: an untrained outfit head on the pretrained Swin backbone, a frozen-backbone Swin baseline with only the outfit head trained, and the final fully fine-tuned Swin model. This makes the training story clearer: no task training -> head-only training -> full fine-tuning.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from outfit_inference import (
    evaluate_checkpoint,
    evaluate_pretrained_backbone_baseline,
    image_dir_for_checkpoint,
    load_checkpoint,
    load_split,
    predict_file,
    random_test_image,
)

pd.set_option("display.max_columns", 80)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


## Configuration

In [2]:
CONFIG = {
    "final_checkpoint": "checkpoints/swin_base_384_final512_512_best.pth",
    "frozen_baseline_checkpoint": "checkpoints/swin_base_384_frozen_baseline_384_best.pth",
    "test_csv": "results/test_split.csv",
    "dataset_dir": "dataset",
    "batch_size": 4,
    "device": None,

    # Baseline: same pretrained Swin backbone, but no outfit training.
    # The 24-label outfit head is freshly initialized with this seed.
    "untrained_baseline_seed": 42,
}

for key in ["final_checkpoint", "frozen_baseline_checkpoint", "test_csv"]:
    path = Path(CONFIG[key])
    if not path.is_file():
        raise FileNotFoundError(f"Missing required file for {key}: {path}")

print("Final checkpoint:", CONFIG["final_checkpoint"])
print("Frozen-backbone baseline checkpoint:", CONFIG["frozen_baseline_checkpoint"])
print("Untrained baseline: pretrained Swin backbone + random outfit head")
print("Test split:", CONFIG["test_csv"])


Final checkpoint: checkpoints/swin_base_384_final512_512_best.pth
Frozen-backbone baseline checkpoint: checkpoints/swin_base_384_frozen_baseline_384_best.pth
Untrained baseline: pretrained Swin backbone + random outfit head
Test split: results/test_split.csv


## Checkpoint Sanity Check

In [3]:
model, checkpoint, device = load_checkpoint(CONFIG["final_checkpoint"], device=CONFIG["device"])
test_df, test_classes = load_split(CONFIG["test_csv"])

print("Device:", device)
print("Model:", checkpoint.get("model_name"))
print("Timm backbone:", checkpoint["timm_name"])
print("Image size:", checkpoint["image_size"])
print("Threshold:", checkpoint.get("threshold", 0.5))
print("Checkpoint classes:", len(checkpoint["class_names"]))
print("Test CSV classes:", len(test_classes))

if list(checkpoint["class_names"]) != list(test_classes):
    raise ValueError("Checkpoint classes do not match test CSV columns. Do not trust evaluation until this is fixed.")

image_dir = image_dir_for_checkpoint(checkpoint, CONFIG["dataset_dir"])
missing = [name for name in test_df["filename"] if not (image_dir / name).is_file()]
print("Image directory:", image_dir)
print("Test rows:", len(test_df))
print("Missing images:", len(missing))
if missing:
    print("First missing:", missing[0])


Device: cuda
Model: swin_base_384_final512
Timm backbone: swin_base_patch4_window12_384.ms_in22k_ft_in1k
Image size: 512
Threshold: 0.5
Checkpoint classes: 24
Test CSV classes: 24
Image directory: dataset/train_512
Test rows: 641
Missing images: 0


## One-Image Smoke Test

In [4]:
image_path, true_labels = random_test_image(checkpoint, CONFIG["test_csv"], CONFIG["dataset_dir"])
result = predict_file(model, checkpoint, image_path, device=device)

print("Image:", image_path)
print("True labels:", ", ".join(true_labels))
print("Top predictions:")
display(pd.DataFrame(result["predictions"]).head(10))


Image: dataset/train_512/9a25d18135e558fc7c2c8e55c956c9b8.jpg
True labels: blazer, bag
Top predictions:


,label,probability,predicted,threshold
0,bag,0.993311,True,0.56
1,belt,0.497270,True,0.41
2,blazer,0.379705,True,0.29
3,heels,0.335668,False,0.53
4,shirt,0.127844,False,0.50
5,shorts,0.119286,False,0.67
6,shoes,0.109605,False,0.44
7,trousers,0.089729,False,0.35
8,jacket,0.075109,False,0.65
9,jeans,0.054968,False,0.88


## Full Test Evaluation

In [5]:
rows = []
per_class_rows = []

final_metrics = evaluate_checkpoint(
    CONFIG["final_checkpoint"],
    split_csv=CONFIG["test_csv"],
    dataset_dir=CONFIG["dataset_dir"],
    batch_size=CONFIG["batch_size"],
    device=device,
)

frozen_baseline_metrics = evaluate_checkpoint(
    CONFIG["frozen_baseline_checkpoint"],
    split_csv=CONFIG["test_csv"],
    dataset_dir=CONFIG["dataset_dir"],
    batch_size=CONFIG["batch_size"],
    device=device,
)

untrained_baseline_metrics = evaluate_pretrained_backbone_baseline(
    checkpoint_path=CONFIG["final_checkpoint"],
    split_csv=CONFIG["test_csv"],
    dataset_dir=CONFIG["dataset_dir"],
    batch_size=CONFIG["batch_size"],
    device=device,
    seed=CONFIG["untrained_baseline_seed"],
)

metric_columns = ["model_name", "image_size", "threshold", "samples", "micro_f1", "macro_f1", "samples_f1", "precision_micro", "recall_micro", "hamming_loss"]
for metrics in [final_metrics, frozen_baseline_metrics, untrained_baseline_metrics]:
    rows.append({key: metrics.get(key) for key in metric_columns})
    per_class_rows.append(pd.Series(metrics["per_class_f1"], name=metrics["model_name"]))

comparison = pd.DataFrame(rows)
comparison = comparison.sort_values("macro_f1", ascending=False).reset_index(drop=True)
comparison.to_csv(RESULTS_DIR / "inference_test_comparison.csv", index=False)
display(comparison)


Evaluating swin_base_384_final512:   0%|          | 0/161 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Per-Class Training Impact

In [ ]:
if per_class_rows:
    per_class_df = pd.DataFrame(per_class_rows)
    per_class_df.to_csv(RESULTS_DIR / "inference_test_per_class_f1.csv")
    display(per_class_df.T.sort_values(final_metrics["model_name"]).head(10))

    plt.figure(figsize=(max(12, len(per_class_df.columns) * 0.55), max(3.5, len(per_class_df) * 0.7)))
    sns.heatmap(per_class_df, vmin=0, vmax=1, cmap="viridis", annot=True, fmt=".2f")
    plt.title("Per-Class Test F1")
    plt.xlabel("Class")
    plt.ylabel("Model")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "inference_test_per_class_f1.png", dpi=180, bbox_inches="tight")
    plt.show()


## Result Interpretation

In [ ]:
final_row = comparison[comparison["model_name"] == final_metrics["model_name"]].iloc[0]
frozen_row = comparison[comparison["model_name"] == frozen_baseline_metrics["model_name"]].iloc[0]
untrained_row = comparison[comparison["model_name"] == untrained_baseline_metrics["model_name"]].iloc[0]

print(f"Final fine-tuned test macro F1: {final_row['macro_f1']:.4f}")
print(f"Frozen-backbone baseline macro F1: {frozen_row['macro_f1']:.4f}")
print(f"Untrained-head baseline macro F1: {untrained_row['macro_f1']:.4f}")
print(f"Fine-tuning gain vs frozen baseline: macro F1 {final_row['macro_f1'] - frozen_row['macro_f1']:+.4f}, micro F1 {final_row['micro_f1'] - frozen_row['micro_f1']:+.4f}")
print(f"Full training gain vs untrained baseline: macro F1 {final_row['macro_f1'] - untrained_row['macro_f1']:+.4f}, micro F1 {final_row['micro_f1'] - untrained_row['micro_f1']:+.4f}")

print("Saved:")
print("-", RESULTS_DIR / "inference_test_comparison.csv")
print("-", RESULTS_DIR / "inference_test_per_class_f1.csv")
print("-", RESULTS_DIR / "inference_test_per_class_f1.png")
